# Introduction

Welcome to the code of the Auto-I3rab project! This notebook is meant to display the joint two-layer HMM that is the focus of the JP.

The layout separates it into four major steps:


*   Preload: Loading all neccesary libraries and datasets


*   Layer 1: Building and Evaluating the Parts-of-speech Hidden Markov Model



*   Layer 2: Building the Simple Word-Mark Hidden Markov Model

*   Joint Viterbi: The heart of this model! This is the joint layer decoder, which builds on layer 1 and 2

There is also an additional section at the end to display all evaluations/scoring metrics once more





# Preload

All libraries needed for this project are loaded here

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
from collections import Counter
from tqdm.notebook import tqdm
import numpy as np
import math
import pickle
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict
from itertools import chain
import re

The datasets can be loaded from the github repository this is linked to.

In [ ]:
# Load in the path for the folder dataset here
POS_data_path = ''

second_data_test_path = ''
second_data_train_path = ''
second_data_val_path = ''

We then process the datasets using their own dedicated functions

In [ ]:
# Dedicated reading function for dataset 1
def read_files_sent(folder_path):
  p = Path(folder_path)
  sentences = []

  for file_path in p.iterdir():
    if file_path.is_file():
      sentence = []

      with open(file_path, 'r') as f:
        for line in f:
          line = line.strip()
          if not line:
            continue

          word, tag = line.rsplit('/', 1)
          sentence.append((word, tag))

          if word == '.' and tag == 'PUNC':
            sentences.append(sentence)
            sentence = []

        if sentence:
          sentences.append(sentence)

  return sentences

In [ ]:
def read_file_second(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [ ]:
ARABIC_DIACRITICS = {
    '\u064b',  # tanween fath
    '\u064c',  # tanween damm
    '\u064d',  # tanween kasr
    '\u064e',  # fatha
    '\u064f',  # damma
    '\u0650',  # kasra
    '\u0651',  # shadda
    '\u0652',  # sukun
}

def extract_word_and_final_letter_mark(word):
    base_letters = []
    final_letter_marks = None
    current_letter_marks = []

    for ch in word:
        if ch in ARABIC_DIACRITICS:
            current_letter_marks.append(ch)
        else:
            if base_letters:
                final_letter_marks = current_letter_marks
            base_letters.append(ch)
            current_letter_marks = []

    if base_letters:
        final_letter_marks = current_letter_marks

    clean_word = ''.join(base_letters)
    last_mark = final_letter_marks[-1] if final_letter_marks else ""

    return (clean_word, last_mark)

def split_into_sentences(text):
    parts = re.split(r'[.!?؟]+\s*', text)
    return [p.strip() for p in parts if p.strip()]

def tokenize_sentence(sentence):
    return re.findall(r'[\u0600-\u06FF]+', sentence)

def process_raw_corpus(text):
    sentences = split_into_sentences(text)
    corpus = []

    for sentence in sentences:
        words = tokenize_sentence(sentence)
        processed_sentence = [
            extract_word_and_final_letter_mark(word)
            for word in words
        ]
        if processed_sentence:
            corpus.append(processed_sentence)

    return corpus

In [ ]:
POS_Data = read_files_sent(POS_data_path)

In [ ]:
second_test = read_file_second(second_data_test_path)
second_train = read_file_second(second_data_train_path)
second_val = read_file_second(second_data_val_path)

Dataset 2 is already split, but dataset 1 isn't, so we split it into train and validation data.

The split here is an 80/20 split, meaning 80% of the data is training and 20% is validation.

The second dataset is processed properly, cleaning up much of the diacritical marks.

In [ ]:
POS_train, POS_val = train_test_split(POS_Data, test_size = 0.2)


In [ ]:
sec_test_full = process_raw_corpus(second_test)
sec_train_full = process_raw_corpus(second_train)
sec_val_full = process_raw_corpus(second_val)

# Layer One: POS HMM

We now begin with the first layer, which is the POS-HMM layer

## Create the Parts-of-Speech Hidden Markov Model based off the first dataset

In [ ]:
def train_HMM(sentences, k):
    init_counts = {}
    trans_counts = {}
    emit_counts = {}
    tag_counts = {}
    tags = set()
    vocab = set()
    num_sentences = 0

    for s in sentences:
      if not s:
        continue

      num_sentences += 1

      first_word, first_tag = s[0]
      init_counts[first_tag] = init_counts.get(first_tag, 0) + 1

      for i, (word, tag) in enumerate(s):
        tags.add(tag)
        vocab.add(word)

        tag_counts[tag] = tag_counts.get(tag, 0) + 1
        emit_counts[(tag, word)] = emit_counts.get((tag, word), 0) + 1

        if i > 0:
          prev_tag = s[i-1][1]
          trans_counts[(prev_tag, tag)] = trans_counts.get((prev_tag, tag), 0) + 1

    tags = sorted(tags)
    vocab = sorted(vocab)

    init_prob = {}
    for tag in tags:
      init_prob[tag] = (init_counts.get(tag, 0) + k) / (num_sentences + (k * len(tags)))

    trans_prob = {}
    for prev_tag in tags:
      total = 0
      for next_tag in tags:
        total += trans_counts.get((prev_tag, next_tag), 0)

      for next_tag in tags:
        if total > 0:
          trans_prob[(prev_tag, next_tag)] = (trans_counts.get((prev_tag,
                                                               next_tag), 0) + k) / (total + (k * len(tags)))

        else:
          trans_prob[(prev_tag, next_tag)] = 0.0

    emit_prob = {}
    for tag in tags:
      total = tag_counts[tag]

      for word in vocab:
        emit_prob[(tag, word)] = (emit_counts.get((tag, word), 0) + k) / (total + (k * len(vocab)))

      emit_prob[(tag, "<UNK>")] = k / (total + (len(vocab) * k))

    return {
        "init_prob": init_prob,
        "trans_prob": trans_prob,
        "emit_prob": emit_prob,
        "tags": tags,
        "vocab": vocab
    }

In [ ]:
first_POS_Model = train_HMM(POS_train, 0.001)

## Implement simple Viterbi decode

In [ ]:
def split_helper(data):
    obs = [row[0] for row in data]
    tags = [row[1] for row in data]

    return obs, tags

In [ ]:
def viterbi_decode(val, model):
    init_prob = model["init_prob"]
    trans_prob = model["trans_prob"]
    emiss_prob = model["emit_prob"]
    tags = model["tags"]
    s, _ = split_helper(val)
    vocab = set(model["vocab"])

    T = len(s)
    V = [{} for _ in range(T)]
    back_point = [{} for _ in range(T)]

    if not s:
      return []

    first_obs = s[0] if s[0] in vocab else "<UNK>"

    for state in tags:
      V[0][state] = math.log(max(init_prob.get(state, 1e-12), 1e-12)) + math.log(max(emiss_prob.get((state, first_obs), 1e-12), 1e-12))
      back_point[0][state] = None

    for t in range(1, T):
      word = s[t] if s[t] in vocab else "<UNK>"

      for curr_state in tags:
        max_prob = float("-inf")
        top_prev_state = None

        # for prev_state in init_prob:
        for prev_state in tags:
          prob = V[t-1][prev_state] + math.log(max(trans_prob.get((prev_state, curr_state), 1e-12), 1e-12)) + math.log(max(emiss_prob.get((curr_state, word), 1e-12), 1e-12))

          if prob > max_prob:
            max_prob = prob
            top_prev_state = prev_state

        V[t][curr_state] = max_prob
        back_point[t][curr_state] = top_prev_state

    top_last_state = max(V[T-1], key = V[T-1].get)
    vi_path = [top_last_state]

    for t in range(T-1, 0, -1):
      vi_path.insert(0, back_point[t][vi_path[0]])

    return vi_path

## Run Viterbi and Evaluate Accuracy

In [ ]:
def evaluate_viterbi(val, model):
  correct = 0
  total = 0
  correct_known = 0
  total_known = 0
  correct_unk = 0
  total_unk = 0

  POS_pred = []
  vocab = set(model["vocab"])

  for sentence in tqdm(val, desc = "Viterbi Decode Process"):
    words, truth = split_helper(sentence)
    pred_tags = viterbi_decode(sentence, model)
    POS_pred.append(pred_tags)

    for word, pred, gold in zip(words, pred_tags, truth):
      if pred == gold:
        correct += 1
      total += 1

      if word in vocab:
        if pred == gold:
          correct_known += 1
        total_known += 1
      else:
        if pred == gold:
          correct_unk += 1
        total_unk += 1


  acc = correct / total if total > 0 else 0.0
  known_acc = correct_known / total_known if total_known else 0
  unk_acc =  correct_unk / total_unk if total_unk else 0

  spec_acc = {
      "acc": acc,
      "known": known_acc,
      "unk": unk_acc
  }

  return POS_pred, spec_acc

In [ ]:
POS_pred, spec_acc = evaluate_viterbi(POS_val, POS_Model)
POS_train_pred, spec_acc_train = evaluate_viterbi(POS_train, POS_Model)

In [ ]:
print("For training set:")
print("Known accuracy:", spec_acc_train["known"])
print("UNK accuracy:", spec_acc_train["unk"])
print("Overall accuracy:", spec_acc_train["acc"])
print("\n\n---------------------------------------------\n")

print("For validation Set:")
print("Known accuracy:", spec_acc["known"])
print("UNK accuracy:", spec_acc["unk"])
print("Overall accuracy:", spec_acc["acc"])

## Check F-1 Scores and Confusion Matrix

In [ ]:
def f_1(data, pred):
  y_true = []
  y_pred = []

  for sentence, pred_tag in zip(data, pred):
    truth = [tag for _, tag in sentence]
    y_true.extend(truth)
    y_pred.extend(pred_tag)


  micro_f = f1_score(y_true, y_pred, average = "micro")
  macro_f = f1_score(y_true, y_pred, average = "macro")
  weight_f = f1_score(y_true, y_pred, average = "weighted")

  f_1_val = {
      "micro": micro_f,
      "macro": macro_f,
      "weighted": weight_f
  }

  return f_1_val

In [ ]:
def confusion(data, pred, labels):
  y_true = []
  y_pred = []

  for sentence, pred_tag in zip(data, pred):
    truth = [tag for _, tag in sentence]
    y_true.extend(truth)
    y_pred.extend(pred_tag)

  c = confusion_matrix(y_true, y_pred, labels = labels)

  return c

In [ ]:
def flatten_labels(labels):
    flat = []

    for item in labels:
        if isinstance(item, list):
            for x in item:
                if isinstance(x, tuple):
                    flat.append(x[1])
                else:
                    flat.append(x)
        else:
            if isinstance(item, tuple):
                flat.append(item[1])
            else:
                flat.append(item)

    return flat

def plot_top_k_confusion_matrix(y_true, y_pred, k=10):

    y_true = flatten_labels(y_true)
    y_pred = flatten_labels(y_pred)

    print("len true:", len(y_true))
    print("len pred:", len(y_pred))
    print("sample true:", y_true[:10])
    print("sample pred:", y_pred[:10])

    assert len(y_true) == len(y_pred), "y_true and y_pred have different lengths"

    tag_counts = Counter(y_true)
    top_k_tags = [tag for tag, _ in tag_counts.most_common(k)]

    filtered_true = []
    filtered_pred = []

    for t, p in zip(y_true, y_pred):
        if t in top_k_tags:
            filtered_true.append(t)
            filtered_pred.append(p if p in top_k_tags else "OTHER")

    labels = top_k_tags + ["OTHER"]

    cm = confusion_matrix(filtered_true, filtered_pred, labels=labels, normalize = 'true')

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(xticks_rotation=45,values_format=".2f")
    plt.title(f"Top {k} POS Tags Confusion Matrix")
    plt.show()

In [ ]:
def disp_cm(cm, labels, title):
  fig_t, ax_t = plt.subplots(figsize = (30, 30))
  disp_t = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = labels)
  disp_t.plot(ax = ax_t)

  plt.title(title)
  plt.xticks(rotation = 90)
  plt.show()

Run the functions to obtain the F-1 Scores, Confusion Matrix, and Top 10 Confusion Matrix

In [ ]:
f_1_val = f_1(POS_val, POS_pred)
con_val = confusion(POS_val, POS_pred, first_POS_Model["tags"])

f_1_train = f_1(POS_train, POS_train_pred)
con_train = confusion(POS_train, POS_train_pred, first_POS_Model["tags"])

Print Everything out to View

In [ ]:
print("For training set:")
print("Micro F-1 Score: ", f_1_train["micro"])
print("Macro F-1 Score:", f_1_train["macro"])
print("Weighted F-1 Score", f_1_train["weighted"])

print("\n\n---------------------------------------------\n")
print("For validation Set:")
print("Micro F-1 Score: ", f_1_val["micro"])
print("Macro F-1 Score:", f_1_val["macro"])
print("Weighted F-1 Score", f_1_val["weighted"])

In [ ]:
disp_cm(con_train, first_POS_Model["tags"], "Confusion Matrix- Training")
disp_cm(con_val, first_POS_Model["tags"], "Confusion Matrix - Validation")

In [ ]:
plot_top_k_confusion_matrix(POS_train, POS_train_pred)
plot_top_k_confusion_matrix(POS_val, POS_pred)

## Markov Chain Illustration

In [ ]:
def draw_HMM_states(model, threshold=0.1, weed = True):
    G = nx.DiGraph()

    tags = model["tags"]
    trans_prob = model["trans_prob"]
    init_prob = model["init_prob"]

    if weed:
      state_counts = Counter(init_prob)
      top_tags = set(tag for tag, _ in state_counts.most_common(10))

      tags = [tag for tag in model["tags"] if tag in top_tags]
      for tag in tags:
        G.add_node(tag)

      for (prev_tag, next_tag), prob in trans_prob.items():
        if prev_tag in top_tags and next_tag in top_tags and prob > threshold:
          G.add_edge(prev_tag, next_tag, weight=prob)
    else:
      for tag in tags:
        G.add_node(tag)
      for (prev_tag, next_tag), prob in trans_prob.items():
        if prob > threshold:
            G.add_edge(prev_tag, next_tag, weight=prob)

    pos = nx.circular_layout(G, scale = 5)

    # 14, 14
    if weed:
      plt.figure(figsize=(6, 6))
      nx.draw(
        G, pos,
        with_labels=True,
        node_size=2500,
        font_size=7,
        arrows=True,)
    else:
      plt.figure(figsize=(14, 14))
      nx.draw(
        G, pos,
        with_labels=True,
        node_size=2500,
        font_size=7,
        arrows=True,)




    edge_labels = {
        (u, v): f"{d['weight']:.2f}"
        for u, v, d in G.edges(data=True)
    }
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

    for tag, (x, y) in pos.items():
        plt.text(x, y + 0.08, f"start={init_prob.get(tag, 0):.2f}", ha="center", fontsize = 7)

    plt.title("Hidden Markov Model: Hidden States and Transitions")
    plt.axis("off")
    plt.show()

In [ ]:
draw_HMM_states(POS_Model, weed = False)
draw_HMM_states(POS_Model)

# Layer 2: Word-Mark HMM

We create a basic word-mark HMM using the function defined in Layer 1

In [ ]:
harakat_model = train_HMM(sec_train_full, 0.001)

In [ ]:
draw_HMM_states(harakat_model, threshold = 0.02, weed = False)

We then retrain the first layer model

In [ ]:
def full_viterbi_2(val, model):
  vocab = set(model["vocab"])
  POS_pred = []

  for sentence in tqdm(val, desc = "Viterbi Decode Process"):
    words, truth = split_helper(sentence)
    pred_tags = viterbi_decode(sentence, model)
    POS_pred.append(pred_tags)

  return POS_pred

In [ ]:
def combine_pos_with_word_mark(pos_data, word_mark_data):
    combined = []

    if len(pos_data) != len(word_mark_data):
        raise ValueError("Number of sentences does not match")

    for pos_sent, wm_sent in zip(pos_data, word_mark_data):
        if len(pos_sent) != len(wm_sent):
            raise ValueError("Sentence lengths do not match")

        merged_sent = []

        for pos, (word, mark) in zip(pos_sent, wm_sent):
            merged_sent.append((word, pos, mark))

        combined.append(merged_sent)

    return combined

In [ ]:
def word_POS_extract(data):
  word_mark_data = [[(word, pos) for (word, pos, mark) in sentence] for sentence in data]

  return word_mark_data

In [2]:
sec_test_POS = full_viterbi_2(sec_test_full, first_POS_Model)
sec_train_POS = full_viterbi_2(sec_train_full, first_POS_Model)
sec_val_POS = full_viterbi_2(sec_val_full, first_POS_Model)

In [ ]:
comb_test = combine_pos_with_word_mark(sec_test_POS, sec_test_full)
comb_train = combine_pos_with_word_mark(sec_train_POS, sec_train_full)
comb_val = combine_pos_with_word_mark(sec_val_POS, sec_val_full)

In [ ]:
word_mark_test = word_POS_extract(comb_test)
word_mark_train = word_POS_extract(comb_train)
word_mark_val = word_POS_extract(comb_val)

In [ ]:
final_POS_Model = train_HMM(word_mark_train, 0.001)

# Joint Layer: Joint Viterbi Decode

In [ ]:
def build_mark_priors(train_data, alpha=1):

    mark_counts = defaultdict(int)
    total = 0

    for sentence in train_data:
        for _, _, mark in sentence:
            mark_counts[mark] += 1
            total += 1

    V = len(mark_counts)

    mark_priors = {
        m: (mark_counts[m] + alpha) / (total + alpha * V)
        for m in mark_counts
    }

    return mark_priors

In [ ]:
def build_mark_given_pos(train_data, all_marks, alpha = 0.1):

    counts = defaultdict(lambda: defaultdict(int))
    pos_counts = defaultdict(int)

    for sentence in train_data:
        for _, pos, mark in sentence:
            counts[pos][mark] += 1
            pos_counts[pos] += 1

    mark_given_pos = {}

    for pos in pos_counts:
        mark_given_pos[pos] = {}
        total = pos_counts[pos] + alpha * len(all_marks)

        for mark in all_marks:
            count = counts[pos][mark] + alpha
            mark_given_pos[pos][mark] = count / total

    return mark_given_pos

In [ ]:
mark_priors = build_mark_priors(comb_train)
mark_given_pos = build_mark_given_pos(comb_train, harakat_model["tags"], alpha = 0.1)

In [ ]:
def viterbi_word_pos_to_mark_test(
    sentence_words,
    sentence_pos,
    transition_probs,   # dict[prev_mark][curr_mark] = P(curr_mark | prev_mark)
    start_probs,        # dict[mark] = P(mark at start)
    emission_probs,     # dict[mark][word] = P(word | mark)
    mark_priors,        # dict[mark] = P(mark)
    mark_given_pos,     # dict[pos][mark] = P(mark | pos)
    all_marks,
    alpha=0.7,
    eps=1e-10
):

    n = len(sentence_words)
    if n == 0:
        return []

    # dp[t][m] = best log score ending in mark m at position t
    dp = [{} for _ in range(n)]
    backptr = [{} for _ in range(n)]

    def joint_log_score(word, pos, mark):
      p_w_given_m = emission_probs.get((mark, word), eps)
      p_m_given_pos = mark_given_pos.get(pos, {}).get(mark, eps)

      # log change
      return alpha * math.log(p_w_given_m + eps) + (1- alpha) * math.log(p_m_given_pos + eps)

    # --- initialization ---
    word0 = sentence_words[0]
    pos0 = sentence_pos[0]

    for m in all_marks:
        # log change
        log_start = math.log(start_probs.get(m, eps) + eps)
        log_emit = joint_log_score(word0, pos0, m)
        dp[0][m] = log_start + log_emit
        backptr[0][m] = None

    # --- recursion ---
    for t in range(1, n):
        word = sentence_words[t]
        pos = sentence_pos[t]

        for curr_mark in all_marks:
            # log change
            log_emit = joint_log_score(word, pos, curr_mark)

            best_prev = None
            best_score = float("-inf")

            for prev_mark in all_marks:
                # log change
                log_trans = math.log(transition_probs.get(prev_mark, {}).get(curr_mark, eps) + eps)

                score = dp[t - 1][prev_mark] + log_trans + log_emit

                if score > best_score:
                    best_score = score
                    best_prev = prev_mark

            dp[t][curr_mark] = best_score
            backptr[t][curr_mark] = best_prev

    # --- termination ---
    best_last_mark = max(dp[n - 1], key=dp[n - 1].get)

    # --- backtrace ---
    best_marks = [best_last_mark]
    for t in range(n - 1, 0, -1):
        best_marks.append(backptr[t][best_marks[-1]])

    best_marks.reverse()
    return best_marks

In [ ]:
def evaluate_word_mark_accuracy(
    dataset,
    decoder_fn,
    train_vocab,
    transition_probs,
    start_probs,
    emission_probs,
    mark_priors,
    mark_given_pos,
    all_marks,
    alpha=0.7):

    total = 0
    correct = 0

    known_total = 0
    known_correct = 0

    unknown_total = 0
    unknown_correct = 0
    total_pred = []

    for sentence in tqdm(dataset, desc = "Joint Viterbi Decode"):
        words = [word for word, pos, gold_mark in sentence]
        pos_tags = [pos for word, pos, gold_mark in sentence]
        gold_marks = [gold_mark for word, pos, gold_mark in sentence]

        pred_marks = decoder_fn(
            sentence_words=words,
            sentence_pos=pos_tags,
            transition_probs=transition_probs,
            start_probs=start_probs,
            emission_probs=emission_probs,
            mark_priors=mark_priors,
            mark_given_pos=mark_given_pos,
            all_marks=all_marks,
            alpha=alpha
        )

        total_pred.append(pred_marks)

        for word, gold, pred in zip(words, gold_marks, pred_marks):
            total += 1
            if pred == gold:
                correct += 1

            if word in train_vocab:
                known_total += 1
                if pred == gold:
                    known_correct += 1
            else:
                unknown_total += 1
                if pred == gold:
                    unknown_correct += 1

    overall_acc = correct / total if total > 0 else 0.0
    known_acc = known_correct / known_total if known_total > 0 else 0.0
    unknown_acc = unknown_correct / unknown_total if unknown_total > 0 else 0.0

    return {
        "pred": total_pred,
        "overall_accuracy": overall_acc,
        "known_accuracy": known_acc,
        "unknown_accuracy": unknown_acc,
        "total_tokens": total,
        "known_tokens": known_total,
        "unknown_tokens": unknown_total
    }

RUN EVERYTHING

In [ ]:
results_test_all = []
results_train_all = []
results_val_all = []

In [5]:
for alpha in tqdm(range(11), desc = "Alpha Hyperparameter Analysis"):
  results_test = evaluate_word_mark_accuracy(
    dataset = comb_test,
    decoder_fn = viterbi_word_pos_to_mark_test,   # ← THIS is decoder_fn
    train_vocab = final_POS_Model["vocab"],
    transition_probs = harakat_model["trans_prob"],
    start_probs = harakat_model["init_prob"],
    emission_probs = harakat_model["emit_prob"],
    mark_priors = mark_priors,
    mark_given_pos = mark_given_pos,
    all_marks = harakat_model["tags"],
    alpha = 0.1 * alpha)

  print("Alpha = ", alpha * 0.1)
  print("Overall Accuracy:", results_test["overall_accuracy"])
  print("Known Accuracy: ", results_test["known_accuracy"])
  print("Unknown Accuracy: ", results_test["unknown_accuracy"])
  print("-------------------------------------------------------\n")
  results_test_all.append(results_test)

In [ ]:
for alpha in tqdm(range(11), desc = "Alpha Hyperparameter Analysis"):
  results_train = evaluate_word_mark_accuracy(
    dataset = comb_train,
    decoder_fn = viterbi_word_pos_to_mark_test,   # ← THIS is decoder_fn
    train_vocab = final_POS_Model["vocab"],
    transition_probs = harakat_model["trans_prob"],
    start_probs = harakat_model["init_prob"],
    emission_probs = harakat_model["emit_prob"],
    mark_priors = mark_priors,
    mark_given_pos = mark_given_pos,
    all_marks = harakat_model["tags"],
    alpha = 0.1 * alpha)

  print("Alpha = ", alpha * 0.1)
  print("Overall Accuracy:", results_train["overall_accuracy"])
  print("Known Accuracy: ", results_train["known_accuracy"])
  print("Unknown Accuracy: ", results_train["unknown_accuracy"])
  print("-------------------------------------------------------\n")
  results_train_all.append(results_train)

In [ ]:
for alpha in tqdm(range(11), desc = "Alpha Hyperparameter Analysis"):
  results_val = evaluate_word_mark_accuracy(
    dataset = comb_val,
    decoder_fn = viterbi_word_pos_to_mark_test,   # ← THIS is decoder_fn
    train_vocab = final_POS_Model["vocab"],
    transition_probs = harakat_model["trans_prob"],
    start_probs = harakat_model["init_prob"],
    emission_probs = harakat_model["emit_prob"],
    mark_priors = mark_priors,
    mark_given_pos = mark_given_pos,
    all_marks = harakat_model["tags"],
    alpha = 0.1 * alpha)

  print("Alpha = ", alpha * 0.1)
  print("Overall Accuracy:", results_val["overall_accuracy"])
  print("Known Accuracy: ", results_val["known_accuracy"])
  print("Unknown Accuracy: ", results_val["unknown_accuracy"])
  print("-------------------------------------------------------\n")
  results_val_all.append(results_val)

Run Evaluations

In [ ]:
marks_test = [[mark for (_, _, mark) in sentence] for sentence in comb_test]
marks_train = [[mark for (_, _, mark) in sentence] for sentence in comb_train]
marks_val = [[mark for (_, _, mark) in sentence] for sentence in comb_val]

In [ ]:
test_f1 = []
train_f1 = []
val_f1 = []

In [ ]:
test_cm = []
train_cm = []
val_cm = []

In [ ]:
for i in range(len(results_test_all)):
  y_pred = [item for sublist in results_test_all[i]["pred"] for item in sublist]
  y_true = [item for sublist in marks_test[:100] for item in sublist]

  micro_f = f1_score(y_true, y_pred, average = "micro")
  macro_f = f1_score(y_true, y_pred, average = "macro")
  weight_f = f1_score(y_true, y_pred, average = "weighted")

  f1 = {
      "micro": micro_f,
      "macro": macro_f,
      "weight": weight_f
  }

  test_f1.append(f1)

In [ ]:
for i in range(len(results_train_all)):
  y_pred = [item for sublist in results_train_all[i]["pred"] for item in sublist]
  y_true = [item for sublist in marks_train[:100] for item in sublist]

  micro_f = f1_score(y_true, y_pred, average = "micro")
  macro_f = f1_score(y_true, y_pred, average = "macro")
  weight_f = f1_score(y_true, y_pred, average = "weighted")

  f1 = {
      "micro": micro_f,
      "macro": macro_f,
      "weight": weight_f
  }

  train_f1.append(f1)

In [ ]:
for i in range(len(results_val_all)):
  y_pred = [item for sublist in results_val_all[i]["pred"] for item in sublist]
  y_true = [item for sublist in marks_val[:100] for item in sublist]

  micro_f = f1_score(y_true, y_pred, average = "micro")
  macro_f = f1_score(y_true, y_pred, average = "macro")
  weight_f = f1_score(y_true, y_pred, average = "weighted")

  f1 = {
      "micro": micro_f,
      "macro": macro_f,
      "weight": weight_f
  }

  val_f1.append(f1)

In [ ]:
for i in range(11):
  y_pred = [item for sublist in results_test_all[i]["pred"] for item in sublist]
  y_true = [item for sublist in marks_test for item in sublist]
  labels = sorted(set(y_true) | set(y_pred))

  cm = confusion_matrix(y_true, y_pred, labels=labels, normalize = 'true')
  disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = labels)

  test_cm.append(disp)

In [ ]:
for i in range(11):
  y_pred = [item for sublist in results_train_all[i]["pred"] for item in sublist]
  y_true = [item for sublist in marks_test for item in sublist]
  labels = sorted(set(y_true) | set(y_pred))

  cm = confusion_matrix(y_true, y_pred, labels=labels, normalize = 'true')
  disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = labels)

  train_cm.append(disp)

In [ ]:
for i in range(11):
  y_pred = [item for sublist in results_val_all[i]["pred"] for item in sublist]
  y_true = [item for sublist in marks_test for item in sublist]
  labels = sorted(set(y_true) | set(y_pred))

  cm = confusion_matrix(y_true, y_pred, labels=labels, normalize = 'true')
  disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = labels)

  val_cm.append(disp)

In [ ]:
for i in range(11):
  print("-------------------------------------")
  print("-------------------------------------\n")
  print("Alpha: ", i * 0.1)

  print("-------------------------------------\n")
  print("F-1 Test\n")
  print("Micro: ", test_f1[i]["micro"])
  print("Macro: ", test_f1[i]["macro"])
  print("Weighted: ", test_f1[i]["weight"])
  print("-------------------------------------\n")

  print("F-1 Train\n")
  print("Micro: ", train_f1[i]["micro"])
  print("Macro: ", train_f1[i]["macro"])
  print("Weighted: ", train_f1[i]["weight"])
  print("-------------------------------------\n")

  print("F-1 Val\n")
  print("Micro: ", val_f1[i]["micro"])
  print("Macro: ", val_f1[i]["macro"])
  print("Weighted: ", val_f1[i]["weight"])
  print("-------------------------------------\n\n")

In [ ]:
for i in range(11):
  print("-------------------------------------")
  print("-------------------------------------\n")
  print("Alpha: ", i * 0.1)

  print("-------------------------------------\n")
  print("F-1 Test\n")
  disp = test_cm[i]
  disp.plot(xticks_rotation=90)
  plt.title("Joint Confusion Matrix - Test")
  plt.show()
  print("-------------------------------------\n")

  print("F-1 Train\n")
  disp = train_cm[i]
  disp.plot(xticks_rotation=90)
  plt.title("Joint Confusion Matrix - Train")
  plt.show()
  print("-------------------------------------\n")

  print("F-1 Val\n")
  disp = val_cm[i]
  disp.plot(xticks_rotation=90)
  plt.title("Joint Confusion Matrix - Validation")
  plt.show()
  print("-------------------------------------\n\n")